## Etapa Adicional — Threshold Fixo para Produção (Deploy)

O corte de 28% definido neste notebook foi aplicado sobre o ranking de
scores de um lote inteiro (top-k% mais anômalos). Em produção, uma API
recebe uma transação por vez, onde esse conceito de "top 28% de um lote"
não se aplica. Esta etapa calcula e persiste um **threshold fixo de
score de anomalia**, equivalente ao percentil 72 do treino (100% - 28%),
que será usado pela API para decidir se uma transação individual avança
ou não ao Stage 2.

In [2]:
# ----------------------------------------------------------------
# CÁLCULO DO THRESHOLD FIXO DE ANOMALIA — para uso em produção
# ----------------------------------------------------------------
# Em batch (notebooks), o corte de 28% foi aplicado sobre o RANKING
# de scores de um lote inteiro (np.argsort + top-k%). Em produção,
# uma API recebe UMA transação por vez — não existe "top 28%" de
# uma transação isolada. A solução é fixar o valor de score de
# anomalia que CORRESPONDIA a esse corte no conjunto de treino, e
# usar esse valor como threshold fixo daqui em diante.

import pandas as pd
import numpy as np
import joblib
import json

caminho_dados   = "/content/drive/MyDrive/fraud-detection-two-stage/data/processed"
caminho_modelos = "/content/drive/MyDrive/fraud-detection-two-stage/models"

df_treino = pd.read_parquet(f"{caminho_dados}/train_processed.parquet")
iso_forest = joblib.load(f"{caminho_modelos}/isolation_forest_stage1.pkl")

features_stage1 = [col for col in df_treino.columns if col not in ["Class", "transaction_id"]]

scores_treino = -iso_forest.score_samples(df_treino[features_stage1])

# O corte de 28% mantém as transações com score ACIMA do percentil 72
# (100% - 28% = 72%) — esse é o valor fixo que vamos usar em produção
PERCENTIL_CORTE = 100 - 28
ANOMALY_SCORE_THRESHOLD = np.percentile(scores_treino, PERCENTIL_CORTE)

print(f"Threshold fixo de score de anomalia (percentil {PERCENTIL_CORTE}): "
      f"{ANOMALY_SCORE_THRESHOLD:.6f}")

# Checagem de sanidade: aplicar esse threshold fixo no treino deve
# reter aproximadamente 28% das transações (pequena diferença é
# esperada, já que percentil exato vs. top-k% podem arredondar diferente)
pct_retido_real = (scores_treino >= ANOMALY_SCORE_THRESHOLD).mean() * 100
print(f"Checagem: aplicando esse threshold no treino retém {pct_retido_real:.2f}% das transações")

# Persiste esse valor junto aos metadados do Stage 1
with open(f"{caminho_modelos}/stage1_metadata.json") as f:
    metadados_stage1 = json.load(f)

metadados_stage1["anomaly_score_threshold_producao"] = float(ANOMALY_SCORE_THRESHOLD)

with open(f"{caminho_modelos}/stage1_metadata.json", "w") as f:
    json.dump(metadados_stage1, f, indent=2)

print(f"\nThreshold salvo em stage1_metadata.json")

Threshold fixo de score de anomalia (percentil 72): 0.423888
Checagem: aplicando esse threshold no treino retém 28.00% das transações

Threshold salvo em stage1_metadata.json
